# 64 — Référence externe : la recherche de LightRAG sur notre graphe

Le pilote du notebook 63 a montré que l'extraction de LightRAG n'est pas praticable avec qwen2.5:7b sur un GPU T4 gratuit : sur 10 documents, 5 ont dépassé la limite de temps de LightRAG, les 5 autres ont pris en moyenne 3,5 minutes chacun (projection : 28 heures pour 475), et les relations extraites plaçaient souvent un verbe à la place de l'entité cible.

Ce notebook conserve **tout le pipeline de LightRAG** — découpage, fusion des entités, index vectoriels, stockage, recherche — et ne remplace qu'une étape : l'extraction. Quand LightRAG demande au modèle les relations d'un résumé, il reçoit les nôtres, dans le format qu'il attend. Le graphe qu'il construit est alors exactement le nôtre : 792 entités, 856 paires reliées. **La comparaison porte donc sur la méthode de recherche seule, à graphe identique.**

Tout le reste passe par qwen2.5:7b, comme dans LightRAG ordinaire : l'extraction des mots-clés de chaque question et le résumé des descriptions d'entités fusionnées.

Mêmes réglages que le notebook 63 : qwen2.5:7b à température 0, encodeur S-PubMedBert-MS-MARCO, un résumé par fragment, reclassement désactivé. Les documents sont traités un par un, pour éviter une collision interne de LightRAG observée en test. Deux modes sont évalués : `mix` (par défaut) et `hybrid` (graphe seul).

**Reprise après coupure** : relancer toutes les cellules. Les documents déjà traités ne sont pas refaits.

**Entrées** : `corpus.json`, `jeu_de_test.json`, `courbe_rappel.json`, `relations_nettoyees.json`
**Sortie** : `baseline_lightrag.json`, à placer dans `data/processed/`

## Cellule 1 — Ollama : installation et serveur

In [1]:
import subprocess, time, requests, os

def ollama_repond():
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

if not os.path.exists("/usr/local/bin/ollama"):
    print("Installation des prérequis et d'Ollama (~2 min)...")
    subprocess.run("apt-get install -y -qq zstd pciutils lshw", shell=True)
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)

if not ollama_repond():
    print("Démarrage du serveur...")
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                     start_new_session=True)
    for _ in range(30):
        if ollama_repond():
            break
        time.sleep(1)

print("Ollama :", "actif" if ollama_repond() else "ARRÊTÉ")

Installation des prérequis et d'Ollama (~2 min)...
Démarrage du serveur...
Ollama : actif


## Cellule 2 — Le modèle

In [2]:
!ollama pull qwen2.5:7b
!ollama list


NAME          ID              SIZE      MODIFIED               
qwen2.5:7b    845dbda0ea48    4.7 GB    Less than a second ago    


## Cellule 3 — LightRAG, version figée

In [3]:
!pip install -q lightrag-hku==1.5.7 sentence-transformers ollama
import lightrag
print("LightRAG", lightrag.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.3/90.3 kB 9.5 MB/s eta 0:00:00
LightRAG 1.5.7


## Cellule 4 — Drive et fichiers

In [4]:
from google.colab import drive, files
import json, os, shutil

drive.mount("/content/drive")
DOSSIER = "/content/drive/MyDrive/stage-graphrag"
os.makedirs(DOSSIER, exist_ok=True)

NECESSAIRES = ["corpus.json", "jeu_de_test.json", "courbe_rappel.json",
               "relations_nettoyees.json"]
manquants = [n for n in NECESSAIRES if not os.path.exists(f"{DOSSIER}/{n}")]
if manquants:
    print("À téléverser :", manquants)
    for nom in files.upload():
        shutil.copy(nom, f"{DOSSIER}/{nom}")
for nom in NECESSAIRES:
    print(f"{nom:<26}", "présent" if os.path.exists(f"{DOSSIER}/{nom}") else "ABSENT")

Mounted at /content/drive
À téléverser : ['relations_nettoyees.json']


Saving relations_nettoyees.json to relations_nettoyees.json
corpus.json                présent
jeu_de_test.json           présent
courbe_rappel.json         présent
relations_nettoyees.json   présent


## Cellule 5 — Notre extraction, à la place de celle de LightRAG

In [5]:
import re
from collections import Counter, defaultdict
from lightrag.utils import normalize_entity_name

with open(f"{DOSSIER}/corpus.json", encoding="utf-8") as f:
    corpus = json.load(f)
with open(f"{DOSSIER}/jeu_de_test.json", encoding="utf-8") as f:
    questions = json.load(f)
with open(f"{DOSSIER}/relations_nettoyees.json", encoding="utf-8") as f:
    relations = json.load(f)

# Même identité des entités que notre pipeline (scripts 48 et 57) :
# deux écritures d'une même entité fusionnent dans un seul nœud.
GRECQUES = {"α": "alpha", "β": "beta", "γ": "gamma", "δ": "delta", "ε": "epsilon",
            "ζ": "zeta", "η": "eta", "θ": "theta", "ι": "iota", "κ": "kappa",
            "λ": "lambda", "μ": "mu", "ν": "nu", "ξ": "xi", "π": "pi", "ρ": "rho",
            "σ": "sigma", "τ": "tau", "φ": "phi", "χ": "chi", "ψ": "psi", "ω": "omega"}

def canon(texte):
    t = texte.lower().strip()
    for g, l in GRECQUES.items():
        t = t.replace(g, l)
    t = re.sub(r"\s*\([^)]*\)\s*", " ", t)
    t = re.sub(r"[^a-z0-9 \-]+", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    t = re.sub(r"[\s\-]+", "", t)
    return t[:-1] if t.endswith("s") and len(t) > 4 else t

# Nom affiché d'une entité : son écriture la plus fréquente dans nos relations
formes, types = defaultdict(Counter), defaultdict(Counter)
for r in relations:
    for cote in ("source", "target"):
        k = canon(r[cote])
        if k:
            formes[k][r[cote].strip()] += 1
            types[k][r[f"{cote}_type"]] += 1
nom = {k: c.most_common(1)[0][0] for k, c in formes.items()}

# Pour chaque résumé, ses relations écrites au format de sortie de LightRAG
S = "<|#|>"
par_doc = defaultdict(lambda: {"entites": {}, "relations": []})
gardees = ecartees = 0
for r in relations:
    s, t = canon(r["source"]), canon(r["target"])
    if not s or not t or s == t or normalize_entity_name(nom[s]) == normalize_entity_name(nom[t]):
        ecartees += 1
        continue
    gardees += 1
    phrase = f"{r['source'].strip()} {r['relation']} {r['target'].strip()}."
    d = par_doc[r["pmid"]]
    for k in (s, t):
        d["entites"].setdefault(k, [])
        if phrase not in d["entites"][k]:
            d["entites"][k].append(phrase)
    d["relations"].append(f"relation{S}{nom[s]}{S}{nom[t]}{S}{r['relation']}{S}{phrase}")

ENREGISTREMENTS = {}
for pmid, d in par_doc.items():
    lignes = [f"entity{S}{nom[k]}{S}{types[k].most_common(1)[0][0]}{S}{' '.join(ph)}"
              for k, ph in d["entites"].items()]
    ENREGISTREMENTS[pmid] = "\n".join(lignes + d["relations"])

def cle_texte(t):
    return re.sub(r"\s+", " ", t).strip()[:300]

PMID_DU_TEXTE = {cle_texte(a["titre"] + "\n\n" + a["resume"]): a["pmid"] for a in corpus}
print(f"Relations gardées : {gardees}   écartées (boucles) : {ecartees}")
print(f"Résumés porteurs d'au moins une relation : {len(ENREGISTREMENTS)} sur {len(corpus)}")

Relations gardées : 932   écartées (boucles) : 3
Résumés porteurs d'au moins une relation : 290 sur 475


## Cellule 6 — Configuration de LightRAG

In [6]:
import lightrag
from sentence_transformers import SentenceTransformer
from lightrag import LightRAG, QueryParam
from lightrag.llm.ollama import ollama_model_complete
from lightrag.utils import EmbeddingFunc

TRAVAIL = f"{DOSSIER}/lightrag_notre_graphe"
os.makedirs(TRAVAIL, exist_ok=True)

# Le modèle de langue vu par LightRAG. L'extraction renvoie notre graphe au
# lieu d'interroger qwen ; tout le reste - mots-clés des questions, résumés
# des descriptions fusionnées - passe par qwen2.5:7b, comme d'ordinaire.
APPELS = Counter()

async def modele(prompt, system_prompt=None, history_messages=[], **kwargs):
    if "---Input Text---" in prompt:
        APPELS["extraction"] += 1
        texte = prompt.split("---Input Text---", 1)[1].split("```")[1]
        pmid = PMID_DU_TEXTE.get(cle_texte(texte))
        if pmid is None:
            APPELS["texte non reconnu"] += 1
        return ENREGISTREMENTS.get(pmid, "") + "\n<|COMPLETE|>"
    if "Based on the last extraction task" in prompt:
        APPELS["glanage"] += 1
        return "<|COMPLETE|>"
    APPELS["qwen2.5:7b"] += 1
    return await ollama_model_complete(prompt, system_prompt=system_prompt,
                                       history_messages=history_messages, **kwargs)

encodeur = SentenceTransformer("pritamdeka/S-PubMedBert-MS-MARCO", device="cuda")

async def encoder(textes, **kwargs):
    return encodeur.encode(list(textes), normalize_embeddings=True, convert_to_numpy=True)

rag = LightRAG(
    working_dir=TRAVAIL,
    llm_model_func=modele,
    llm_model_name="qwen2.5:7b",
    llm_model_kwargs={"host": "http://127.0.0.1:11434", "timeout": 900,
                      "options": {"num_ctx": 16384, "temperature": 0,
                                  "num_predict": 1024}},   # plafonne toute réponse emballée
    embedding_func=EmbeddingFunc(embedding_dim=768, func=encoder),
    chunk_token_size=4000,
    max_parallel_insert=1,   # un document à la fois : évite une collision interne de LightRAG
)
await rag.initialize_storages()
print("LightRAG prêt")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

INFO: [] Created new empty graph file: /content/drive/MyDrive/stage-graphrag/lightrag_notre_graphe/graph_chunk_entity_relation.graphml
INFO: Role LLM Configuration (initialized):
INFO:  - extract: None/None, host=None, max_async=4, timeout=240
INFO:  - keyword: None/None, host=None, max_async=4, timeout=240
INFO:  - query: None/None, host=None, max_async=4, timeout=240
INFO:  - vlm: None/None, host=None, max_async=4, timeout=240
INFO: [] Process 1085 KV load full_docs with 0 records
INFO: [] Process 1085 KV load text_chunks with 0 records
INFO: [] Process 1085 KV load full_entities with 0 records
INFO: [] Process 1085 KV load full_relations with 0 records
INFO: [] Process 1085 KV load entity_chunks with 0 records
INFO: [] Process 1085 KV load relation_chunks with 0 records
INFO: [] Process 1085 KV load llm_response_cache with 0 records
INFO: [] Process 1085 doc status load doc_status with 0 records


LightRAG prêt


## Cellule 7 — Insertion des 475 résumés

In [7]:
import time
import networkx as nx
from lightrag.base import DocStatus

# Reprise : un document déjà connu de LightRAG n'est jamais réinséré.
connus = await rag.doc_status.get_docs_by_statuses(list(DocStatus))
deja = await rag.doc_status.get_docs_by_statuses([DocStatus.PROCESSED])
nouveaux = [a for a in corpus if a["pmid"] not in connus]
en_suspens = len(connus) - len(deja)
print(f"Déjà traités : {len(deja)}   en suspens : {en_suspens}   nouveaux : {len(nouveaux)}")

debut = time.time()
if nouveaux:
    await rag.ainsert([a["titre"] + "\n\n" + a["resume"] for a in nouveaux],
                      ids=[a["pmid"] for a in nouveaux],
                      file_paths=[a["pmid"] for a in nouveaux])
elif en_suspens:
    await rag.apipeline_process_enqueue_documents()
for tentative in range(3):
    if not await rag.doc_status.get_docs_by_statuses([DocStatus.FAILED]):
        break
    print("Nouvelle tentative pour les documents en échec...")
    await rag.apipeline_process_enqueue_documents()
print(f"Durée : {(time.time() - debut) / 60:.1f} min")

traites = await rag.doc_status.get_docs_by_statuses([DocStatus.PROCESSED])
echoues = await rag.doc_status.get_docs_by_statuses([DocStatus.FAILED])
print(f"Traités : {len(traites)} / {len(corpus)}   échoués : {len(echoues)}")
for pm, st in list(echoues.items())[:5]:
    print("  ÉCHEC", pm, str(getattr(st, "error_msg", st))[:160])
print("Appels :", dict(APPELS))

g = nx.read_graphml(f"{TRAVAIL}/graph_chunk_entity_relation.graphml")
print(f"\nGraphe dans LightRAG : {g.number_of_nodes()} entités, {g.number_of_edges()} relations")
print("Notre graphe          : 792 entités, 856 paires reliées")

Déjà traités : 0   en suspens : 0   nouveaux : 475


Streaming output truncated to the last 5000 lines.
INFO: Completed merging: 0 entities, 0 extra entities, 0 relations
INFO: [] chunks flush: embedding 1 vectors in 1 batch(es) (batch_num=10)
INFO: [] Writing graph with 500 nodes, 521 edges
INFO: In memory DB persist to disk
INFO: Completed processing file 225/475: 19829090
INFO: Extracting stage 226/475: 16610572
INFO: Processing d-id: 16610572
INFO: Chunking F(legacy): size=4000, split_only=False, overlap=100, doc_id: 16610572
INFO: Parsing (native): 38736624
INFO:  == LLM cache == saving: default:extract:6009e59532736f1563a20e1539dfb051
INFO:  == LLM cache == saving: default:extract:0ef2605efa31f6fca89e123e5d0d08fb
INFO: Chunk 1 of 1 extracted 0 Ent + 0 Rel 16610572-chunk-000
INFO: Merging stage 226/475: 16610572
INFO: Phase 0: Writing recovery indexes for 16610572: 0 candidate entities, 0 candidate relations
INFO: Phase 1: Processing 0 entities from 16610572 (async: 8)
INFO: Phase 2: Processing 0 relations from 16610572 (async: 8)
I

Durée : 13.4 min
Traités : 475 / 475   échoués : 0
Appels : {'extraction': 475, 'glanage': 475, 'qwen2.5:7b': 29}

Graphe dans LightRAG : 792 entités, 856 relations
Notre graphe          : 792 entités, 856 paires reliées


## Cellule 8 — Rappel de LightRAG sur notre graphe

In [8]:
BUDGETS = (1, 3, 5, 10, 20)
MODES = ("mix", "hybrid")

CLASSEMENTS = {m: {} for m in MODES}
for q in questions:
    for mode in MODES:
        r = await rag.aquery_data(q["question"], param=QueryParam(mode=mode, enable_rerank=False))
        vus = []
        for c in r.get("data", {}).get("chunks", []):
            pm = str(c.get("file_path", ""))
            if pm and pm not in vus:
                vus.append(pm)
        CLASSEMENTS[mode][q["id"]] = vus

def rappel(cl, K, categorie=None):
    qs = [q for q in questions if categorie is None or q["type"] == categorie]
    return sum(len(set(q["pmids"]) & set(cl[q["id"]][:K])) / len(q["pmids"]) for q in qs) / len(qs)

for m in MODES:
    n = sorted(len(v) for v in CLASSEMENTS[m].values())
    print(f"Mode {m:<7} documents renvoyés par question : médiane {n[len(n) // 2]}, min {n[0]}, max {n[-1]}")

with open(f"{DOSSIER}/courbe_rappel.json", encoding="utf-8") as f:
    GRAPHE = {l["id"]: l["graphe"] for l in json.load(f)["detail"]}
colonnes = [(f"LightRAG {m}", CLASSEMENTS[m]) for m in MODES] + [("notre recherche", GRAPHE)]
print(f"\n{'K':>13}" + "".join(f"{n:>17}" for n, _ in colonnes))
for K in BUDGETS:
    print(f"{K:>13}" + "".join(f"{rappel(c, K):>17.2f}" for _, c in colonnes))
print("\nRappel@5 par catégorie")
for cat in ("single-hop", "pont", "intersection"):
    print(f"{cat:>13}" + "".join(f"{rappel(c, 5, cat):>17.2f}" for _, c in colonnes))

INFO: keyword LLM func: 4 new workers initialized (Timeouts: Func: 240s, Worker: 480s, Health Check: 495s)
INFO:  == LLM cache == saving: mix:keywords:790dfee0c392519f5d19a29880697c53
INFO: Query edges: pharmacokinetic ratio, enzyme, paroxetine (top_k:40, cosine:0.2)
INFO: Global query: 40 entites, 40 relations
INFO: Naive query: 20 chunks (chunk_top_k:20 cosine:0.2)
INFO: Raw search results: 40 entities, 40 relations, 20 vector chunks
INFO: After truncation: 40 entities, 40 relations
INFO: Selecting 92 from 92 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 40 relations
INFO: Round-robin merged chunks: 112 -> 100 (deduplicated 12)
INFO: Final context: 40 entities, 40 relations, 20 chunks
INFO: Final chunks S+F/O: E2/1 C1/2 E1/2 C1/3 E3/3 C1/4 E1/4 E14/5 E3/6 E2/7 C1/8 E2/8 E2/9 E2/10 E1/11 C1/12 E2/12 E1/13 C1/14 E1/14
INFO:  == LLM cache == saving: hybrid:keywords:894a940afc77cda63f852debb59d5df3
INFO: Query edges: pharmacokinetic rat

Mode mix     documents renvoyés par question : médiane 20, min 20, max 20
Mode hybrid  documents renvoyés par question : médiane 20, min 20, max 20

            K     LightRAG mix  LightRAG hybrid  notre recherche
            1             0.38             0.40             0.45
            3             0.53             0.53             0.78
            5             0.55             0.57             0.83
           10             0.59             0.68             0.88
           20             0.76             0.80             0.95

Rappel@5 par catégorie
   single-hop             0.67             0.67             1.00
         pont             0.50             0.50             0.67
 intersection             0.44             0.48             0.71


## Cellule 9 — Enregistrer et télécharger

In [9]:
resultat = {"lightrag": lightrag.__version__, "modele": "qwen2.5:7b",
            "extraction": "relations_nettoyees.json (la nôtre)",
            "documents_traites": len(traites), "appels": dict(APPELS),
            "classements": CLASSEMENTS}
with open("baseline_lightrag.json", "w", encoding="utf-8") as f:
    json.dump(resultat, f, ensure_ascii=False, indent=1)
shutil.copy("baseline_lightrag.json", f"{DOSSIER}/baseline_lightrag.json")
files.download("baseline_lightrag.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>